# Experiment 14 — Staging on RGB + 1 Monocular Relative-Depth Channel — Colab Run

Trains **MiT-B1 + CORN, ImageNet weights** (task_3's confirmed best config) on 4-channel samples: the full, uncropped RGB photo (channels 0-2) plus 1 per-image min-max normalized monocular relative-depth channel (channel 3, Depth Anything V2 Small, already generated at `datasets/wound-stage-depth-anything-v2-small/` for WP5). No Task 1/Task 2 dependency, no masking or cropping — see `scripts/generate_dataset.py`'s docstring. Same original PIID 3-fold split as task_3's own ablation (verified directly via `build_folds()`). config_id = `mit_b1__corn__exp14_depth_channel`.

This is the first experiment in this chapter that actually **trains a network** on RGB+depth — every earlier depth result (WP5) was a hand-fit threshold rule on depth summary statistics, not a trained model.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the dataset

Upload `wound-staging-rgbd-exp-14.zip` to `drive/MyDrive/thesis/` first (built locally by `more_experiments/exp/exp14_depth_channel/scripts/generate_dataset.py` — no GPU needed for that step, it only stacks an existing RGB jpeg with an existing depth `.npy`; ~190MB zipped, ~210MB extracted — 4-channel `.npy` arrays, not JPEGs).

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-staging-rgbd-exp-14.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-staging-rgbd-exp-14"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
source = candidates[0]
print("dataset found at:", source)

os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(source, DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Install dependencies

Same `requirements.txt` as Task 3 — `scripts/train.py` reuses `task_3/src`'s engine/losses/metrics/splits/augmentations/utils directly; only `model.py` and `dataset.py` (4-channel versions) are local.

In [ ]:
import os
!pip install -q -r {REPO_DIR}/task_3/requirements.txt
os.chdir(f"{REPO_DIR}/more_experiments/exp/exp14_depth_channel/scripts")
print(os.getcwd())

## 5. Sanity-check the model before trusting any training run

Verifies the zero-init property, gradient flow into the depth channel, and output shapes directly (not assumed) — same checks as every other `exp/*` 4/8-channel model in this chapter.

In [ ]:
!python test_4ch_model.py

## 6. Redirect checkpoints/results to Drive

So a killed/disconnected runtime doesn't lose completed-fold history or checkpoints.

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/exp14_depth_channel_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/results.csv"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 7. Train — MiT-B1 + CORN, RGB+depth 4-channel input, 3-fold

Single config, no grid. No `--batch` override: uses `configs/base.yaml`'s `batch_size: 16`, matching task_3's and exp-3/3.1's own baseline ablation run exactly. `train.py` already skips completed `(config, fold)` rows and resumes a killed fold from its last saved epoch; this loop restarts the process on a crash. A full runtime disconnect kills the loop too — just re-run this cell, training picks up from the last saved epoch either way.

In [ ]:
import subprocess
import time

cmd = ["python", "-u", "train.py", "--workers", "4"]
while True:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    returncode = proc.wait()
    if returncode == 0:
        print("training finished")
        break
    print(f"train.py exited with code {returncode}, retrying in 10s...")
    time.sleep(10)

## 8. Check results

3 rows expected (one per fold) once training completes. Compare directly against the RGB baseline (accuracy 0.8168 ± 0.0150, macro-F1 0.8182 ± 0.0138, QWK 0.9138 ± 0.0164 — `task_3/results/task_3_outputs/results/ablation_summary.csv`, `mit_b1__corn` row).

In [ ]:
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/results/results.csv")
display(df)
print("mean test accuracy:", df["accuracy"].mean())
print("mean test macro_f1: ", df["macro_f1"].mean())
print("mean test qwk:      ", df["qwk"].mean())